# Image Preprocessing for Complex Engineering Drawings

This notebook is a **step-by-step practical course** on image preprocessing for industrial and mechanical engineering drawings (for example CAD exports, scanned blueprints, or PDF drawings such as `data/Sample-DWG1.pdf`).

## Why this matters

Engineering drawings differ from plain text documents:

- **Thin linework** (for example 0.25–0.5 mm on paper) becomes fragile after scanning or rasterization.
- **Mixed content**: dimensions, leaders, symbols, title blocks, and sometimes **grids** or colored layers.
- **Large uniform areas** (paper background) and **low local contrast** in some regions.
- **Noise** from scanners (speckle), JPEG compression artifacts, or PDF rasterization settings.

Good preprocessing **stabilizes** line thickness, **separates** ink from background, and **reduces** noise *before* OCR, vectorization, or ML-based detection—often more impactful than swapping algorithms alone.

## Learning outcomes

By working through this notebook you will:

0. Understand the **basics**: pixels, `(y, x)` indexing, coordinate directions, and simple OpenCV drawing on a canvas (Part 1A).
1. Load a **PDF drawing** and convert it to a high-resolution image.
2. Apply **core transforms**: grayscale, resize/DPI awareness, denoising, contrast enhancement, sharpening.
3. Compare **binarization** methods (global Otsu vs. adaptive thresholds, optional Sauvola).
4. Use **morphology** to clean speckle and optionally reconnect broken strokes (with caution).
5. Try **engineering-oriented** ideas: emphasizing thin lines (top-hat), optional **deskew** and **edge maps** for inspection.
6. Build a small **reusable visualization** pattern to compare methods side by side.
7. Read an **intensity histogram** to judge whether global thresholding is plausible.
8. Inspect **HSV channels** when drawings use colored layers (optional workflow).

## How to use this notebook

Run cells **from top to bottom** the first time. Each section introduces one idea, runs code, and explains what you should observe. Adjust parameters (kernel sizes, thresholds) and re-run to build intuition.

---
**Prerequisites:** basic Python only. **No prior image-processing background is assumed** — Part 1A starts from pixels and coordinates; later parts build on that vocabulary.


## Part 1 — Environment setup

Install libraries used in this course:

| Package | Role |
|---------|------|
| `opencv-python` | Image processing (filters, morphology, thresholds) |
| `numpy` | Arrays and math |
| `matplotlib` | Plotting |
| `pymupdf` (`fitz`) | PDF → raster image (high quality) |
| `scikit-image` | Advanced thresholding (Sauvola) |

> **Note:** If installation fails on a restricted network, run the same `pip` command in a terminal with appropriate proxy settings.


In [ ]:
%pip install -q opencv-python numpy matplotlib pymupdf scikit-image


In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import re
import numpy as np
import matplotlib.pyplot as plt
import cv2

try:
    import fitz  # PyMuPDF
except ImportError:
    fitz = None

try:
    from skimage.filters import threshold_sauvola
except ImportError:
    threshold_sauvola = None

# Global output directory for saved figures/images across all parts
OUTPUT_DIR = (Path(".").resolve() / "data" / "preprocessing_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SAVE_COUNTER = 0


def _slugify(text: str) -> str:
    text = (text or "figure").strip().lower()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    return text.strip("_") or "figure"


def save_current_figure(tag: str = "figure", dpi: int = 140):
    """Save the currently active matplotlib figure and print the path."""
    global SAVE_COUNTER
    SAVE_COUNTER += 1
    fname = f"{SAVE_COUNTER:03d}_{_slugify(tag)}.png"
    out_path = OUTPUT_DIR / fname
    plt.savefig(out_path, dpi=dpi, bbox_inches="tight")
    print(f"Saved figure: {out_path}")
    return out_path


print("Libraries loaded successfully.")
print("Global output folder:", OUTPUT_DIR)


## Part 1A — Foundations for complete beginners (pixels, coordinates, drawing)

If you are new to image processing, read this section slowly. The goal is not to memorize APIs yet, but to build a correct mental model:

- A digital image is a **grid of pixels** with numeric values.
- You must be consistent about **`(x, y)` vs `[y, x]`** when indexing.
- Drawing functions help you **visualize** what your code is doing when you debug preprocessing.

The cells below are self-contained on synthetic images (no PDF required). After this foundation, the notebook continues with the engineering drawing pipeline.

### What is a pixel?

A **pixel** (“picture element”) is the **smallest controllable unit** of a raster image. Think of the image as a large table where each cell holds one or more numbers that describe **color** or **brightness** at that location.

Key ideas:

1. **Raster vs vector**
   - A **vector** drawing (many CAD/PDF sources) stores curves and text mathematically; it can be scaled without becoming “blocky.”
   - A **raster** image stores a **fixed grid** of pixels. When you zoom in, you eventually see stair-steps (**aliasing**) because information between pixels is not defined.

2. **Grayscale vs color**
   - A **grayscale** image typically has **one number per pixel**, often an integer `0..255` (`uint8`), where `0` is black and `255` is white (this convention is common but not universal—always check your pipeline).
   - A **color** image in OpenCV is usually **3 channels per pixel**: **B**, **G**, **R** (note the order: **BGR**, not RGB). Three `uint8` values means each pixel can take one of `256^3` colors.

3. **Shape**
   - In NumPy/OpenCV, a color image array commonly has shape **`(height, width, 3)`**.
   - The total number of stored values is `height * width * channels`.

4. **Why pixels matter for preprocessing**
   - Filters like blur or median replace each pixel using **neighborhood pixels** (local window).
   - Thresholding turns each pixel into ink/background using its intensity (and sometimes neighborhood statistics).

Run the next cell to inspect a **tiny** random grayscale image: its `shape`, `dtype`, and how `imshow` maps numbers to gray levels.

In [ ]:
# Pixels as numbers: shape, dtype, and a tiny "image"
rng = np.random.default_rng(0)
tiny = rng.integers(0, 256, size=(6, 10), dtype=np.uint8)

print("tiny image shape (H, W):", tiny.shape)
print("dtype:", tiny.dtype)
print("number of pixels:", tiny.size)
print("memory order is row-major: row 0 is stored first, left-to-right.\n")
print("First row (intensities 0..255):\n", tiny[0])

plt.figure(figsize=(5, 2.2), dpi=110)
plt.imshow(tiny, cmap="gray", vmin=0, vmax=255)
plt.title("Each cell is one pixel (grayscale intensity)")
plt.xlabel("x (column index increases →)")
plt.ylabel("y (row index increases ↓)")
plt.colorbar(label="intensity")
save_current_figure("part1a_tiny_pixel_grid")
plt.show()

### BGR vs RGB (same pixels, different channel order)

OpenCV’s default for color images is **BGR** (blue channel first in memory). Matplotlib’s `imshow` assumes **RGB** ordering.

So if you forget to convert, “red ink” may appear **blue** on screen. The fix is usually:

```python
plt.imshow(cv2.cvtColor(bgr_image, cv2.COLOR_BGR2RGB))
```

The next cell shows the same numpy array displayed **with** and **without** conversion.

In [ ]:
# Same pixel values: OpenCV BGR vs what Matplotlib expects (RGB)
patch = np.zeros((80, 160, 3), dtype=np.uint8)
patch[:, :80] = (0, 0, 255)   # "full red" in BGR: B=0,G=0,R=255
patch[:, 80:] = (255, 0, 0)   # "full blue" in BGR

fig, ax = plt.subplots(1, 2, figsize=(7, 2.8), dpi=110)
ax[0].imshow(patch)  # interprets last dim as RGB -> channels are misread
ax[0].set_title("plt.imshow(patch) - wrong interpretation for OpenCV BGR")
ax[1].imshow(cv2.cvtColor(patch, cv2.COLOR_BGR2RGB))
ax[1].set_title("plt.imshow(cvtColor(..., BGR2RGB)) - correct colors")
for a in ax:
    a.axis("off")
plt.tight_layout()
save_current_figure("part1a_bgr_vs_rgb")
plt.show()

### Overview of the image coordinate system

Digital images are stored on a **2D grid of pixels**. Two conventions collide in practice, so it helps to memorize one sentence:

> **NumPy / OpenCV pixel access uses `image[row, col]` which is `image[y, x]`.**

Meaning:

- **`x` (column index)** increases as you move **to the right**.
- **`y` (row index)** increases as you move **downward** (because the next row is stored below the previous row).

The **origin `(x=0, y=0)`** is the **top-left** corner of the image.

**Why this confuses beginners:** geometry classes often plot `y` upward, but **screen coordinates** follow the monitor convention (top-left origin, `y` down). Matplotlib’s `imshow` uses the same image convention unless you change axis settings.

**OpenCV drawing functions** usually take points as `(x, y)` tuples (column first, then row), for example `cv2.circle(img, (200, 120), ...)`.

**NumPy slicing** for a rectangular crop is:

```python
patch = img[y1:y2, x1:x2]
```

which means “rows from `y1` up to (but not including) `y2`, columns from `x1` up to (but not including) `x2`.”

Run the next cell to visualize the origin direction and a labeled point.

In [ ]:
# Coordinate system demo: rows increase downward; columns increase rightward
H, W = 160, 240
coord_img = np.full((H, W, 3), 255, dtype=np.uint8)

# Origin marker at top-left pixel neighborhood
cv2.circle(coord_img, (0, 0), 6, (0, 0, 255), thickness=-1)
cv2.putText(coord_img, "(0,0) top-left", (8, 18), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 0, 0), 1, cv2.LINE_AA)

# A point at (x=200, y=120) — note cv2.circle uses (x,y)
pt = (200, 120)
cv2.drawMarker(coord_img, pt, (255, 0, 0), markerType=cv2.MARKER_CROSS, markerSize=14, thickness=2)
cv2.putText(coord_img, f"({pt[0]},{pt[1]})", (pt[0] + 8, pt[1] - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 0, 0), 1, cv2.LINE_AA)

# Row scanline at y=100
cv2.line(coord_img, (0, 100), (W - 1, 100), (200, 200, 200), 1)
cv2.putText(coord_img, "row y=100", (6, 96), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (120, 120, 120), 1, cv2.LINE_AA)

plt.figure(figsize=(6, 3.5), dpi=110)
plt.imshow(cv2.cvtColor(coord_img, cv2.COLOR_BGR2RGB))
plt.title("Image coordinates: x increases to the right, y increases downward")
plt.axis("off")
save_current_figure("part1a_coordinate_system")
plt.show()

print("Image shape:", coord_img.shape, "→ (height, width, channels)")
print("Pixel access pattern: image[row_y, col_x] == image[y, x]")

### Accessing and manipulating pixels

**Reading a pixel** at column `x` and row `y`:

```python
b, g, r = image[y, x]
```

**Writing a pixel:**

```python
image[y, x] = (b, g, r)
```

**Working with a region (ROI):** NumPy slicing uses the same `[row_start:row_end, col_start:col_end]` pattern:

```python
patch = image[y1:y2, x1:x2]
image[y1:y2, x1:x2] = 0
```

**Common pitfalls for beginners:**

1. **Wrong order `(x, y)` vs `(y, x)`:** array indexing is `[row, col]` → `[y, x]`. Some OpenCV functions take `(x, y)` tuples; always read the docs for the function you call.
2. **Integer overflow:** if you add brightness using `uint8`, values wrap around. Safer pattern: cast to a larger type, add, then `np.clip`, then cast back to `uint8` (see the code cell).
3. **Modifying in place:** slicing returns a **view** in many cases; assigning through a slice changes the original image. Use `.copy()` when you want an independent duplicate.

In [ ]:
# Accessing and manipulating individual pixels and regions
demo = np.full((120, 200, 3), 230, dtype=np.uint8)  # light gray canvas (BGR)

# 1) Read one pixel at (x, y) = (50, 40)
x, y = 50, 40
b, g, r = demo[y, x]
print(f"Pixel at (x={x}, y={y}) has BGR = ({b}, {g}, {r})")

# 2) Write one pixel (set to red in BGR)
demo[y, x] = (0, 0, 255)

# 3) Fill a rectangular ROI using NumPy slicing [y1:y2, x1:x2]
demo[60:100, 30:170] = (255, 128, 0)  # orange-ish region

# 4) Arithmetic on a region (brighten slightly, then clip)
roi = demo[10:50, 120:190].astype(np.int16) + 25
demo[10:50, 120:190] = np.clip(roi, 0, 255).astype(np.uint8)

plt.figure(figsize=(6, 3), dpi=110)
plt.imshow(cv2.cvtColor(demo, cv2.COLOR_BGR2RGB))
plt.title("Single-pixel write + ROI slice + brighten region")
plt.axis("off")
save_current_figure("part1a_pixel_manipulation")
plt.show()

### Drawing lines, rectangles, and circles (OpenCV)

OpenCV draws **on top of** an existing image array (it modifies pixels in place unless you pass a copy).

Common functions:

| Function | What it draws |
|----------|----------------|
| `cv2.line(img, pt1, pt2, color, thickness)` | Straight segment between two points |
| `cv2.rectangle(img, pt1, pt2, color, thickness)` | Axis-aligned rectangle |
| `cv2.circle(img, center, radius, color, thickness)` | Circle |

**Color in OpenCV:** BGR triple `(Blue, Green, Red)` with values in `0..255` for `uint8` images.

**Thickness:** use a positive integer for stroke width, or `cv2.FILLED` (alias `-1`) for a filled shape.

**Why learn this here?** When you debug pipelines, you often **overlay** bounding boxes, ROI rectangles, or keypoints on the drawing to verify coordinates and preprocessing steps.

In [ ]:
# Drawing primitives on a blank canvas (BGR, uint8)
canvas = np.full((240, 420, 3), 255, dtype=np.uint8)  # white background

# Line: image, pt1, pt2, color (B,G,R), thickness
cv2.line(canvas, (30, 30), (200, 120), (0, 0, 255), thickness=2)

# Rectangle: top-left corner, bottom-right corner
cv2.rectangle(canvas, (220, 40), (380, 140), (0, 128, 0), thickness=2)

# Circle: center, radius
cv2.circle(canvas, (110, 190), radius=40, color=(255, 0, 0), thickness=2)

# Filled shapes: set thickness to cv2.FILLED (or -1)
cv2.circle(canvas, (320, 190), radius=35, color=(200, 200, 0), thickness=-1)

cv2.putText(
    canvas,
    "Line  Rect  Circles",
    (20, 225),
    cv2.FONT_HERSHEY_SIMPLEX,
    0.55,
    (40, 40, 40),
    1,
    cv2.LINE_AA,
)

plt.figure(figsize=(7, 4), dpi=110)
plt.imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))
plt.title("cv2.line, cv2.rectangle, cv2.circle, cv2.putText")
plt.axis("off")
save_current_figure("part1a_drawing_primitives")
plt.show()

**Checkpoint — you should now be able to answer (without scrolling back):**

1. What does `image.shape` represent for a color image?
2. Why do we write `image[y, x]` even though we speak about “pixel (x, y)”?
3. What is the difference between **BGR** and **RGB**, and why does `plt.imshow` often need `cv2.cvtColor(..., cv2.COLOR_BGR2RGB)`?

If any answer feels shaky, re-run the Part 1A cells and tweak the numbers (move the point, change ROI slices, change colors) until it feels obvious.

**Optional hands-on drills (sketch on paper, then code):**

1. On a `200 x 200` white canvas, draw a **red** square outline from `(50,50)` to `(150,150)` using `cv2.rectangle` (remember BGR for red).
2. Set every pixel in the **bottom half** of the image to light gray `(200,200,200)` using slicing `y1:y2, :`.
3. Pick `(x=120, y=80)`, read `image[y, x]`, then set that pixel to pure **green** in BGR.
4. Draw a **filled** circle centered at `(100,100)` with radius `30` in a color of your choice (`thickness=-1`).

**End of Part 1**
---

### Quick catalog — common preprocessing stages for technical drawings

| Stage | Typical goal | Representative OpenCV ideas |
|-------|----------------|------------------------------|
| **Rasterization** | Enough resolution for the thinnest stroke | PyMuPDF `Matrix(zoom, zoom)` |
| **Grayscale / channels** | Reduce dimensionality; optionally isolate colored layers | `cvtColor`, split HSV |
| **Resize** | Faster iteration while prototyping | `resize` + `INTER_AREA` for downscale |
| **Denoise** | Remove speckle without blurring lines | `medianBlur`, `bilateralFilter`, `fastNlMeansDenoising` |
| **Contrast** | Separate ink from uneven background | `CLAHE`, histogram equalization |
| **Sharpen** | Recover edge acuity after smoothing | Unsharp mask |
| **Binarize** | Clean black/white mask for OCR or morphology | Otsu, adaptive, Sauvola |
| **Morphology** | Remove pepper noise, bridge small gaps | opening, closing, small kernels |
| **Structure emphasis** | Highlight thin strokes vs. blobs | Top-hat / black-hat |
| **Geometry / QA** | Deskew, measure edge preservation | rotation search, `Canny` |

The sections below follow this order on `data/Sample-DWG1.pdf`.

## Part 2 — Paths and loading the sample drawing

We use **`data/Sample-DWG1.pdf`**. The next cell defines paths relative to the notebook working directory (usually the folder containing this `.ipynb` file).

**Tip:** In VS Code / Jupyter, set the notebook kernel's working directory to the `Module 5` folder so `data/` resolves correctly.


In [ ]:
BASE_DIR = Path(".").resolve()
PDF_PATH = BASE_DIR / "data" / "Sample-DWG1.pdf"
OUT_DIR = BASE_DIR / "data" / "preprocessing_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

if not PDF_PATH.is_file():
    raise FileNotFoundError(
        f"Expected PDF at {PDF_PATH}. Place Sample-DWG1.pdf under data/ or adjust PDF_PATH."
    )
print("PDF:", PDF_PATH)
print("Outputs will be saved under:", OUT_DIR)


### Convert PDF page to an image (raster)

Vector PDFs must be **rasterized** for pixel-based processing. We use PyMuPDF with a **zoom matrix** so we control effective resolution (similar to choosing DPI).

- `zoom = 2` doubles width and height (four times as many pixels). For OCR or fine linework, `2–3` is common.
- Higher zoom yields more detail and higher memory use.

The function returns a **BGR** image (`uint8`) as used by OpenCV.


In [ ]:
def pdf_page_to_bgr(pdf_path: Path, page_index: int = 0, zoom: float = 2.0) -> np.ndarray:
    # Render one PDF page to a BGR numpy array (OpenCV convention).
    if fitz is None:
        raise RuntimeError("PyMuPDF (fitz) is required: pip install pymupdf")
    doc = fitz.open(pdf_path)
    page = doc.load_page(page_index)
    mat = fitz.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=mat, alpha=False)
    img_rgb = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, pix.n)
    doc.close()
    bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    return bgr


img_bgr = pdf_page_to_bgr(PDF_PATH, page_index=0, zoom=2.0)
h, w = img_bgr.shape[:2]
print(f"Loaded page 0: {w} x {h} pixels (zoom=2.0)")


### Visualization helper

We will compare many variants. The helper below shows one or more images in a row with titles.


In [ ]:
def show_images(images, titles=None, cmap=None, figsize_per=(4, 4), dpi=100, save_prefix="step"):
    # Display one or more grayscale or BGR images in a row, then auto-save both raw outputs and the figure.
    global SAVE_COUNTER
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=(figsize_per[0] * n, figsize_per[1]), dpi=dpi)
    if n == 1:
        axes = [axes]

    titles = titles or [None] * n
    for idx, (ax, im, title) in enumerate(zip(axes, images, titles), start=1):
        if im.ndim == 2:
            ax.imshow(im, cmap=cmap or "gray")
            write_img = im
        else:
            ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
            write_img = im

        ax.set_title(title or "", fontsize=10)
        ax.axis("off")

        # Save each output image in native array form (BGR or grayscale)
        SAVE_COUNTER += 1
        img_name = f"{SAVE_COUNTER:03d}_{_slugify(save_prefix)}_{idx:02d}_{_slugify(title or 'image')}.png"
        img_path = OUTPUT_DIR / img_name
        cv2.imwrite(str(img_path), write_img)

    plt.tight_layout()
    save_current_figure(tag=f"{save_prefix}_figure", dpi=dpi)
    plt.show()


show_images([img_bgr], ["Original (display as RGB)"], save_prefix="part2_original")


## Part 3 — Grayscale and color space

For most **line-drawing** analysis, **luminance** is enough. We convert BGR to grayscale with `cvtColor`.

**When to keep color:** if you need to **split layers** by color (for example red revision marks vs. black geometry), work in HSV or analyze channels before converting.

**Takeaway:** Grayscale reduces data size and simplifies many pipelines.


In [ ]:
gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
show_images([gray], ["Grayscale"], cmap="gray")


### Part 3.1 — Intensity histogram (why thresholding is hard)

A **histogram** counts how many pixels have each intensity (0 = black, 255 = white in our default display convention for grayscale).

For engineering drawings you often see:

- A **large peak** near white (paper background).
- A **smaller bump** or tail toward dark values (linework, text, hatched regions).

If the drawing and background peaks **overlap**, no single global threshold separates them perfectly—this is when **CLAHE**, **adaptive** thresholds, or **Sauvola** help.

Run the next cell to plot the histogram of `gray` and the **cumulative** distribution. Use this as a sanity check before binarization.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5), dpi=110)

ax[0].hist(gray.ravel(), bins=256, range=(0, 256), color="steelblue", alpha=0.85)
ax[0].set_title("Grayscale histogram")
ax[0].set_xlabel("Pixel intensity")
ax[0].set_ylabel("Count")

hist, _ = np.histogram(gray.ravel(), bins=256, range=(0, 256))
cdf = np.cumsum(hist).astype(np.float64)
cdf /= cdf[-1]
ax[1].plot(np.arange(256), cdf, color="darkorange")
ax[1].set_title("Cumulative distribution (CDF)")
ax[1].set_xlabel("Pixel intensity")
ax[1].set_ylabel("Fraction of pixels ≤ intensity")
ax[1].grid(True, alpha=0.3)
plt.tight_layout()
save_current_figure("part3_histogram_cdf")
plt.show()

## Part 4 — Resolution and resizing

If images are too large for quick experimentation, **resize** while preserving aspect ratio. For **downscaling** thin lines, `INTER_AREA` is usually best. For **upscaling**, `INTER_CUBIC` or `INTER_LANCZOS4` are common.

Doubling resolution in software does not create new detail if the source was low-DPI; prefer **rasterizing the PDF at adequate zoom** first (Part 2).


In [ ]:
scale = 0.5  # set to 1.0 for full resolution in later cells
if scale != 1.0:
    small = cv2.resize(gray, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
else:
    small = gray.copy()

print("Shape after resize:", small.shape)
show_images([gray, small], ["Full gray", f"Resized x{scale}"], cmap="gray")


## Part 5 — Denoising (from drawing image to noisy image)

In this section we follow a practical workflow for beginners:

1. Start with the **engineering drawing image** (`small` or `gray`).
2. Create synthetic **noise** on top of that drawing.
3. Apply one denoising method at a time.
4. For each method, display **before vs after** clearly.

### Methods used in this part

| Method | Idea | Typical use |
|--------|------|-------------|
| **Median filter** | Replaces a pixel with the median of its neighborhood | Salt-and-pepper noise |
| **Gaussian blur** | Weighted low-pass smoothing | Random grain noise (but can blur edges) |
| **Bilateral filter** | Smooths while preserving stronger edges | Keep line boundaries better |

For **thin CAD lines**, use small kernels and verify that details are not lost.

In [ ]:
# Step 5.1: start from the engineering drawing image
base_drawing = small  # use `gray` for full-resolution experiments

# Add synthetic noise to imitate real acquisition noise
rng = np.random.default_rng(42)

# (a) Salt-and-pepper noise
sp_prob = 0.015  # fraction of pixels to corrupt
noisy_sp = base_drawing.copy()
rand_map = rng.random(base_drawing.shape)
noisy_sp[rand_map < (sp_prob / 2)] = 0
noisy_sp[rand_map > 1 - (sp_prob / 2)] = 255

# (b) Gaussian noise
gauss_sigma = 12.0
gauss_noise = rng.normal(0, gauss_sigma, base_drawing.shape)
noisy_gauss = np.clip(base_drawing.astype(np.float32) + gauss_noise, 0, 255).astype(np.uint8)

# Build one mixed noisy image for denoising demo (closer to real scans)
noisy_input = cv2.addWeighted(noisy_sp, 0.6, noisy_gauss, 0.4, 0)

show_images(
    [base_drawing, noisy_sp, noisy_gauss, noisy_input],
    ["Base drawing", "Salt & pepper noise", "Gaussian noise", "Mixed noisy input"],
    cmap="gray",
    figsize_per=(3.2, 3.2),
)
print("Use `noisy_input` as the input for denoising filters below.")

### Step 5.2 — Median filter (before vs after)

Median filter is usually the first choice for **salt-and-pepper** noise because it removes isolated black/white spikes while preserving edges better than a simple average blur.

In [ ]:
median_out = cv2.medianBlur(noisy_input, 3)
show_images(
    [noisy_input, median_out],
    ["Before (noisy input)", "After median (ksize=3)"],
    cmap="gray",
    figsize_per=(4, 4),
)

### Step 5.3 — Gaussian blur (before vs after)

Gaussian blur smooths random grain noise effectively, but it can soften thin lines and character edges. Keep kernel size and sigma small for engineering drawings.

Run the next code cell to apply Gaussian blur and compare **before vs after**.


In [ ]:
gaussian_out = cv2.GaussianBlur(noisy_input, (5, 5), sigmaX=1.0)
show_images(
    [noisy_input, gaussian_out],
    ["Before (noisy input)", "After Gaussian blur (5x5, sigma=1.0)"],
    cmap="gray",
    figsize_per=(4, 4),
)

### Step 5.4 — Bilateral filter (before vs after)

Bilateral filter reduces noise while preserving edges better than Gaussian blur, so it is often useful for line drawings where boundaries are important.

In [ ]:
bilateral_out = cv2.bilateralFilter(noisy_input, d=5, sigmaColor=50, sigmaSpace=50)
show_images(
    [noisy_input, bilateral_out],
    ["Before (noisy input)", "After bilateral (d=5, sigma=50)"],
    cmap="gray",
    figsize_per=(4, 4),
)

# Keep a denoised image for the next parts of the notebook
work = bilateral_out.copy()
print("`work` is set to bilateral_out for Part 6 onward.")

## Part 6 — Contrast enhancement (CLAHE)

**CLAHE** (Contrast Limited Adaptive Histogram Equalization) improves local contrast without amplifying noise as aggressively as global histogram equalization in many cases.

Parameters:

- `clipLimit` — higher values increase contrast (can amplify noise).
- `tileGridSize` — size of local regions (for example 8×8).

Useful when lighting or scan intensity **varies across the sheet**.

In [ ]:
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
clahe_img = clahe.apply(work)

global_eq = cv2.equalizeHist(work)

show_images([work, global_eq, clahe_img], ["Input", "Global histogram equalization", "CLAHE"], cmap="gray")


## Part 7 — Sharpening (unsharp mask)

**Unsharp masking** enhances edges by subtracting a blurred version from the original:

`sharp = (1 + amount) * I - amount * blur(I)`

Increase `amount` carefully—**halos** and **OCR errors** can appear on small text and thin strokes.


In [ ]:
def unsharp_mask(gray_u8: np.ndarray, sigma: float = 1.0, amount: float = 1.5) -> np.ndarray:
    blur = cv2.GaussianBlur(gray_u8.astype(np.float32), (0, 0), sigmaX=sigma)
    sharp = (1.0 + amount) * gray_u8.astype(np.float32) - amount * blur
    return np.clip(sharp, 0, 255).astype(np.uint8)


sharp = unsharp_mask(clahe_img, sigma=1.0, amount=1.2)
show_images([clahe_img, sharp], ["CLAHE (input to sharpen)", "Unsharp mask"], cmap="gray")


## Part 8 — Binarization

Many pipelines need a **binary** image (ink vs. background).

### 8.1 Otsu (global)

Automatically selects a threshold when the histogram is roughly **bimodal**.


In [ ]:
# 8.1 Otsu (global threshold)
# Keep one common base image for all thresholding experiments.
base = sharp  # try base = work to compare behavior

# Otsu automatically picks the global threshold value.
_, otsu = cv2.threshold(base, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

show_images([base, otsu], ["Base (uint8)", "Otsu (global)"], cmap="gray", figsize_per=(3.5, 3.5))


### 8.2 Adaptive threshold (local)

Adaptive threshold computes a local threshold per neighborhood, useful when brightness is uneven across the page.

- `blockSize`: local window size (odd number, e.g. 21, 35, 51)
- `C`: constant subtracted from local statistic

Tune these values carefully for thin engineering lines.

In [ ]:
# 8.2 Adaptive threshold (mean and Gaussian)
adaptive_mean = cv2.adaptiveThreshold(
    base, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, blockSize=35, C=10
)
adaptive_gauss = cv2.adaptiveThreshold(
    base, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, blockSize=35, C=10
)

show_images(
    [base, adaptive_mean, adaptive_gauss],
    ["Base (uint8)", "Adaptive mean", "Adaptive Gaussian"],
    cmap="gray",
    figsize_per=(3.2, 3.2),
)

### 8.3 Sauvola (scikit-image)

Sauvola is another local thresholding method often used for document images with non-uniform background. This section runs only if `scikit-image` is installed.

In [ ]:
# 8.3 Sauvola threshold
sauvola_bin = None
if threshold_sauvola is not None:
    f = base.astype(np.float64) / 255.0
    t = threshold_sauvola(f, window_size=25)
    sauvola_bin = (f > t).astype(np.uint8) * 255
    show_images([base, sauvola_bin], ["Base (uint8)", "Sauvola"], cmap="gray", figsize_per=(3.5, 3.5))
else:
    print("scikit-image not available; skipping Sauvola.")

## Part 9 — Morphological operations

Morphology treats bright regions as **foreground** by default. For drawings where lines are **dark** on **light** paper, we often **invert** first so lines become white (foreground), then invert back for display if needed.

| Operation | Effect (on foreground) |
|-----------|-------------------------|
| **Erosion** | Thins foreground, removes small protrusions |
| **Dilation** | Thickens foreground, fills small holes |
| **Opening** | Erosion then dilation — removes small bright noise |
| **Closing** | Dilation then erosion — bridges small gaps |

Use **small structuring elements** (for example 3×3) to avoid changing design geometry.

Below we demonstrate on an **inverted Otsu** image so **lines are white**.


In [ ]:
# Lines dark on paper -> invert so lines are white (255) for morphology
bin_inv = cv2.bitwise_not(otsu)  # paper black, lines white — good as mask foreground

kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
opened = cv2.morphologyEx(bin_inv, cv2.MORPH_OPEN, kernel, iterations=1)
closed = cv2.morphologyEx(bin_inv, cv2.MORPH_CLOSE, kernel, iterations=1)

show_images(
    [bin_inv, opened, closed],
    ["Inverted Otsu (lines=white)", "Opening (noise removal)", "Closing (gap bridging)"],
    cmap="gray",
    figsize_per=(3.5, 3.5),
)


## Part 10 — Emphasizing thin structures (top-hat transform)

The **top-hat** is `image - opening(image)`: it highlights **small bright details** smaller than the structuring element. On an **inverted** drawing (lines as bright), a top-hat can emphasize **thin linework** relative to large filled regions.

This is a classical idea; effectiveness depends on line width vs. kernel size. Experiment with **odd** kernel sizes (15, 21, …).


In [ ]:
# Use CLAHE output; invert so lines are bright
inv = cv2.bitwise_not(clahe_img)
k = 21
kernel_big = cv2.getStructuringElement(cv2.MORPH_RECT, (k, k))
tophat = cv2.morphologyEx(inv, cv2.MORPH_TOPHAT, kernel_big)

show_images([clahe_img, inv, tophat], ["CLAHE", "Inverted (lines bright)", f"Top-hat k={k}"], cmap="gray")


## Part 11 — Deskew (simple projection-based rotation search)

Scanned sheets may be slightly rotated. A **practical** approach for text-heavy pages is to maximize horizontal projection variance over a small angle range. This is a simplified demo—not as robust as full Hough line fitting on long edges, but it is easy to teach.

**Limitation:** Works best when there is a dominant text/line direction. For some pure mechanical views, consider **Hough** or **PCA on contours** instead.


In [ ]:
def rotate_bound(image: np.ndarray, angle_deg: float) -> np.ndarray:
    h, w = image.shape[:2]
    center = (w // 2, h // 2)
    M = cv2.getRotationMatrix2D(center, angle_deg, 1.0)
    return cv2.warpAffine(image, M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)


def estimate_skew_angle(gray_img: np.ndarray, angle_range=(-5.0, 5.0), step=0.25):
    # Angle that maximizes variance of row sums on a binarized rotated image.
    _, b = cv2.threshold(gray_img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    best_angle, best_score = 0.0, -1.0
    for a in np.arange(angle_range[0], angle_range[1] + 1e-6, step):
        r = rotate_bound(b, a)
        proj = np.sum(r, axis=1).astype(np.float64)
        score = np.var(proj)
        if score > best_score:
            best_score, best_angle = score, a
    return best_angle


ang = estimate_skew_angle(work)
print(f"Estimated skew angle (degrees): {ang:.2f}")
deskewed = rotate_bound(work, -ang)
show_images([work, deskewed], ["Before deskew", "After deskew (projection heuristic)"], cmap="gray")


## Part 12 — Edge maps (quality inspection)

**Canny** edge detection is useful to **visualize** line density and to debug whether preprocessing blurred important structure. It is not usually the final binarization for OCR, but it helps **compare** settings.

Parameters `t1` and `t2` are hysteresis thresholds (ratio `t2/t1` often around 2–3).


In [ ]:
edges = cv2.Canny(work, threshold1=50, threshold2=150)
show_images([work, edges], ["Input", "Canny edges"], cmap="gray")


## Part 13 — Optional: save intermediate images

Saving PNGs lets you compare results outside the notebook (or feed them to an OCR notebook).


In [ ]:
cv2.imwrite(str(OUT_DIR / "01_gray.png"), gray)
cv2.imwrite(str(OUT_DIR / "02_clahe.png"), clahe_img)
cv2.imwrite(str(OUT_DIR / "03_otsu.png"), otsu)
print("Wrote sample PNGs to", OUT_DIR)


### Part 13.1 — Industrial drawing specifics (grids, color layers, stamps)

**Reference grids** (regular spacing) can confuse line detectors or OCR because they add many edges. Common mitigation strategies (beyond this notebook) include:

- **Frequency-domain filtering**: suppress narrow-band peaks corresponding to grid spacing in the 2D FFT magnitude spectrum (notch or comb filters), then invert back to the spatial domain.
- **Morphological tricks**: very long linear structuring elements oriented horizontally/vertically can attenuate thin grids if they are thinner than main geometry—but this is **risky** because it can also damage legitimate long centerlines.

**Colored layers** (revision clouds in red, dimensions in blue, etc.) are sometimes easier to separate in **HSV** or **Lab** than in BGR. The next cell splits the drawing into H, S, V channels for exploration (useful when you want to mask one ink color).

**Stamps and logos** in title blocks add high-contrast regions that can dominate histograms. Cropping the **working area** (geometry zone) before thresholding often improves robustness.

These topics are **application-dependent**; always validate on your own drawing standards (ASME, ISO, company CAD exports).

In [ ]:
# HSV channel exploration on the full-color raster (helps when layers use distinct colors)
hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
h_ch, s_ch, v_ch = cv2.split(hsv)

show_images(
    [h_ch, s_ch, v_ch],
    ["H (hue)", "S (saturation)", "V (value / brightness)"],
    cmap="gray",
    figsize_per=(3.5, 3.5),
)

## Part 14 — Summary checklist (industrial drawing preprocessing)

When you design a real pipeline, walk through this list:

1. **Rasterize** the PDF at sufficient zoom/DPI for the smallest stroke you must preserve.
2. **Convert** to grayscale unless color layers matter.
3. **Crop** margins and large empty borders to save compute.
4. **Denoise** lightly; prefer median/small bilateral over heavy blur.
5. **Contrast** with CLAHE if illumination is uneven.
6. **Binarize** with Otsu first; switch to adaptive or Sauvola if the background is not uniform.
7. **Morphology** with small kernels to remove speckle or bridge gaps—verify geometry impact.
8. **Validate** with edges or difference maps before downstream OCR/vectorization.

## Exercises

1. Set `scale = 1.0` and re-run. How do denoising and binarization results change?
2. Sweep `blockSize` and `C` in adaptive threshold; note when thin lines disappear.
3. Replace the sample PDF with another drawing and document which method worked best and why.
4. Combine **deskew → CLAHE → Otsu → opening** and compare OCR or line detection on the result vs. the raw raster.

---
**End of notebook.**


## Part 15 — Deep Learning Image Processing (optional)

This part introduces deep learning in a beginner-friendly way. We will train a **tiny CNN denoiser** using patches generated from the current drawing.

Why this is useful:

- Classical filters (median/gaussian/bilateral) use fixed rules.
- A CNN can **learn** noise-to-clean mapping from data.
- For engineering drawings, learned models can preserve line structures better when tuned well.

> This section is optional and may run slower on CPU. It is designed for learning, not production accuracy.

In [ ]:
# Optional install for deep learning section (uncomment if needed)
# %pip install -q torch torchvision

In [ ]:
# Deep learning imports and runtime setup
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import Dataset, DataLoader
    TORCH_OK = True
except Exception as e:
    TORCH_OK = False
    print("PyTorch is unavailable:", e)

if TORCH_OK:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("PyTorch device:", device)
else:
    device = None

### 15.1 Build a tiny noisy-clean training set from the drawing

We generate random patches from `base_drawing` (or `work`) and synthesize noise.

- **Input**: noisy patches
- **Target**: clean patches

This gives us a self-contained dataset for demonstrating supervised denoising.

In [ ]:
if TORCH_OK:
    # Use drawing image prepared earlier in the notebook
    src_for_dl = base_drawing if 'base_drawing' in globals() else work
    src_for_dl = src_for_dl.astype(np.uint8)

    rng_dl = np.random.default_rng(123)
    patch = 48
    n_samples = 700

    clean_patches = []
    noisy_patches = []

    H, W = src_for_dl.shape
    for _ in range(n_samples):
        y = rng_dl.integers(0, H - patch)
        x = rng_dl.integers(0, W - patch)
        clean = src_for_dl[y:y+patch, x:x+patch].copy()

        # Mix gaussian + salt-pepper noise
        gauss = rng_dl.normal(0, 10.0, clean.shape)
        noisy = np.clip(clean.astype(np.float32) + gauss, 0, 255).astype(np.uint8)

        pm = rng_dl.random(clean.shape)
        noisy[pm < 0.01] = 0
        noisy[pm > 0.99] = 255

        clean_patches.append(clean)
        noisy_patches.append(noisy)

    clean_patches = np.stack(clean_patches)
    noisy_patches = np.stack(noisy_patches)

    # visualize one sample pair
    show_images(
        [clean_patches[0], noisy_patches[0]],
        ["DL target (clean patch)", "DL input (noisy patch)"],
        cmap="gray",
        figsize_per=(3.2, 3.2),
        save_prefix="part15_dataset_sample",
    )
    print("Patch dataset shape:", noisy_patches.shape)
else:
    print("Skip: install torch first to run Part 15.")

### 15.2 Train a compact CNN denoiser

Model: a small fully-convolutional network that predicts a clean patch from a noisy patch.

For teaching purposes we keep it small:

- Runs on CPU in a short time.
- Shows the full deep learning loop: dataset → model → loss → optimizer → training epochs.

In [ ]:
if TORCH_OK:
    class PatchDenoiseDataset(Dataset):
        def __init__(self, noisy_np, clean_np):
            # normalize to [0,1], add channel dimension [N,1,H,W]
            self.x = torch.from_numpy(noisy_np[:, None, :, :].astype(np.float32) / 255.0)
            self.y = torch.from_numpy(clean_np[:, None, :, :].astype(np.float32) / 255.0)

        def __len__(self):
            return self.x.shape[0]

        def __getitem__(self, idx):
            return self.x[idx], self.y[idx]


    class TinyDenoiser(nn.Module):
        def __init__(self):
            super().__init__()
            self.net = nn.Sequential(
                nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(inplace=True),
                nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(inplace=True),
                nn.Conv2d(32, 16, 3, padding=1), nn.ReLU(inplace=True),
                nn.Conv2d(16, 1, 3, padding=1),
            )

        def forward(self, x):
            return torch.sigmoid(self.net(x))


    ds = PatchDenoiseDataset(noisy_patches, clean_patches)
    dl = DataLoader(ds, batch_size=32, shuffle=True)

    model = TinyDenoiser().to(device)
    loss_fn = nn.L1Loss()
    opt = optim.Adam(model.parameters(), lr=1e-3)

    epochs = 4
    loss_history = []
    for ep in range(1, epochs + 1):
        model.train()
        running = 0.0
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            loss = loss_fn(pred, yb)
            opt.zero_grad()
            loss.backward()
            opt.step()
            running += loss.item() * xb.size(0)

        ep_loss = running / len(ds)
        loss_history.append(ep_loss)
        print(f"Epoch {ep}/{epochs} - L1 loss: {ep_loss:.5f}")

    # Save training curve
    plt.figure(figsize=(5.5, 3.2), dpi=120)
    plt.plot(np.arange(1, epochs + 1), loss_history, marker="o")
    plt.xlabel("Epoch")
    plt.ylabel("L1 loss")
    plt.title("TinyDenoiser training loss")
    plt.grid(True, alpha=0.3)
    save_current_figure("part15_training_curve")
    plt.show()
else:
    print("Skip: install torch first to run training.")

### 15.3 Apply the trained model to a full noisy drawing

We run inference on `noisy_input` and compare:

- noisy input
- classical bilateral output
- deep-learning output (tiny CNN)

All outputs are displayed and auto-saved.

In [ ]:
if TORCH_OK:
    model.eval()
    with torch.no_grad():
        inp = torch.from_numpy(noisy_input[None, None, :, :].astype(np.float32) / 255.0).to(device)
        out = model(inp).cpu().numpy()[0, 0]

    dl_denoised = np.clip(out * 255.0, 0, 255).astype(np.uint8)

    ref_bilateral = bilateral_out if 'bilateral_out' in globals() else noisy_input

    show_images(
        [noisy_input, ref_bilateral, dl_denoised],
        ["Noisy input", "Classical bilateral", "Deep CNN denoised"],
        cmap="gray",
        figsize_per=(3.8, 3.8),
        save_prefix="part15_full_image_compare",
    )

    # Keep optional output for downstream experiments
    work_dl = dl_denoised.copy()
    print("Saved deep-learning output in variable: work_dl")
else:
    print("Skip: install torch first to run inference.")